<a href="https://colab.research.google.com/github/aadityane93/Deep_Learning/blob/main/Topic_7_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Text

**Deep Learning with Python, Chapter 11**

---

1. **Standardize** — lowercase, remove punctuation
2. **Tokenize** — split into units (words, N-grams, characters)
3. **Index** — map each token to a unique integer, then encode as a vector

We use Keras's `TextVectorization` layer to handle all three steps automatically.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import TextVectorization
import string
import re

# Tiny corpus
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms."
]

# textvectorization layer
text_vectorization = TextVectorization(output_mode="int")
# adapt()
text_vectorization.adapt(dataset)

vocabulary = text_vectorization.get_vocabulary()
print("Learned Vocabulary:", vocabulary)
print("Index 0 = padding mask | Index 1 = [UNK] (out-of-vocabulary)")

# to show how [UNK]
test_sentence = "I write, rewrite, and still rewrite again"
print(f"\nRaw Text: '{test_sentence}'")

#standardization
standardized_text = test_sentence.lower()
standardized_text = re.sub(f"[{re.escape(string.punctuation)}]", "", standardized_text)
print(f"After Standardization: '{standardized_text}'")

                             #tokenization
print(f"After Tokenization: {standardized_text.split()}")

# indexing
print(f"After Indexing (Final Tensor): {text_vectorization(test_sentence).numpy()}")

Learned Vocabulary: ['', '[UNK]', np.str_('erase'), np.str_('write'), np.str_('then'), np.str_('rewrite'), np.str_('poppy'), np.str_('i'), np.str_('blooms'), np.str_('and'), np.str_('again'), np.str_('a')]
Index 0 = padding mask | Index 1 = [UNK] (out-of-vocabulary)

Raw Text: 'I write, rewrite, and still rewrite again'
After Standardization: 'i write rewrite and still rewrite again'
After Tokenization: ['i', 'write', 'rewrite', 'and', 'still', 'rewrite', 'again']
After Indexing (Final Tensor): [ 7  3  5  9  1  5 10]


## Preparing the IMDB Dataset

**Dataset:** 50,000 movie reviews (positive / negative) from Stanford's IMDB corpus.

We download the raw files, carve out a 20% validation split, then use `text_dataset_from_directory` to create three `tf.data.Dataset` objects: `train_ds`, `val_ds`, `test_ds`.

In [ ]:
import os, pathlib, shutil, random

# Download and unpack the raw IMDB text files
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
# Remove unlabeled reviews
!rm -r aclImdb/train/unsup

base_dir  = pathlib.Path("aclImdb")
val_dir   = base_dir / "val"
train_dir = base_dir / "train"

# 20% of training files as a validation set
# Fixed seed (1337) ensures the same split every run
for category in ("neg", "pos"):
    os.makedirs(val_dir / category, exist_ok=True)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)

# text_dataset_from_directory infers labels from subfolder names (neg=0, pos=1)
batch_size = 32
train_ds = keras.utils.text_dataset_from_directory("aclImdb/train", batch_size=batch_size)
val_ds   = keras.utils.text_dataset_from_directory("aclImdb/val",   batch_size=batch_size)
test_ds  = keras.utils.text_dataset_from_directory("aclImdb/test",  batch_size=batch_size)

# Inspect one batch to confirm the data looks correct before modeling
for inputs, targets in train_ds:
    print("Input shape:", inputs.shape)
    print("Example review:", inputs[0].numpy())
    print("Label (0=Negative, 1=Positive):", targets[0].numpy())
    break

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  57.4M      0  0:00:01  0:00:01 --:--:-- 57.4M
Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.
Input shape: (32,)
Example review: b"As someone who lives near Buffalo, New York, this movie scored points with me before I even saw it, since the story is based here. There are even some bit parts with real-life news-TV anchor people from Buffalo..and, for once, it doesn't knock the area. Hallelujah!<br /><br />Theology-wise, puh-leeze!!! God is still made to look and think like humans...and, of course, be a bit on the liberal side. Being the lightweight comedy it is, it's nothing that should win any awards but it still is entertaining and is a pleasant way to kill 102 minutes. <br /><br />There are some laugh-out-loud slapstick 

## Approach 1 — Bag-of-Words with TF-IDF

The simplest approach: **discard word order** and represent each review as a weighted bag of bigrams.

- **Bigrams** (`ngrams=2`) capture local word pairs (e.g. `"not good"`) to recover some ordering signal.
- **TF-IDF** weighting down-weights common words ("the", "a") and boosts rare, informative ones.
- Each review becomes a single 20,000-dimensional vector → fed into a small Dense network.



In [ ]:
from tensorflow.keras import layers


# max_tokens=20000 keeps only the top 20k
# output_mode='tf_idf' frequency across corpus
text_vectorization_tfidf = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="tf_idf",
)

text_only_train_ds = train_ds.map(lambda x, y: x)
print("Adapting vocabulary...")
text_vectorization_tfidf.adapt(text_only_train_ds)

# num_parallel_calls=4 vectorizes across 4 CPU cores while GPU trains the previous batch
tfidf_2gram_train_ds = train_ds.map(lambda x, y: (text_vectorization_tfidf(x), y), num_parallel_calls=4)
tfidf_2gram_val_ds   = val_ds.map(  lambda x, y: (text_vectorization_tfidf(x), y), num_parallel_calls=4)

for inputs, targets in tfidf_2gram_train_ds:
    print("TF-IDF batch shape:", inputs.shape, "← 32 samples × 20,000 bigram features")
    break

# Simple Dense classifier: input is the 20k-dim TF-IDF vector
# Dropout(0.5) randomly zeros half the neurons each step to prevent overfitting
def get_dense_model():
    inputs  = keras.Input(shape=(20000,))
    x       = layers.Dense(16, activation="relu")(inputs)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model   = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    return model

model_bow = get_dense_model()
model_bow.summary()

# .cache() stores the vectorized batches in memory after the first epoch,
# so we don't re-run the CPU vectorization on every subsequent epoch
print("\nTraining Bag-of-Words model...")
model_bow.fit(tfidf_2gram_train_ds.cache(), validation_data=tfidf_2gram_val_ds.cache(), epochs=3)

Adapting vocabulary...
TF-IDF batch shape: (32, 20000) ← 32 samples × 20,000 bigram features


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)


Training Bag-of-Words model...
Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.7697 - loss: 0.5236 - val_accuracy: 0.8780 - val_loss: 0.3070
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8500 - loss: 0.3590 - val_accuracy: 0.8622 - val_loss: 0.3042
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8698 - loss: 0.3250 - val_accuracy: 0.8528 - val_loss: 0.3163


## Approach 2 — Sequence Model with Word Embeddings + Bidirectional LSTM

- Embedding layer — maps each token to a learned 256-dim vector; similar words cluster nearby. mask_zero=True skips padding.
- Bidirectional LSTM — reads sequences left-to-right and right-to-left for full context.
- Input shape — reviews truncated/padded to 600 tokens → (batch, 600).



In [ ]:
max_length = 600
max_tokens = 20000


text_vectorization_seq = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization_seq.adapt(text_only_train_ds)

int_train_ds = train_ds.map(lambda x, y: (text_vectorization_seq(x), y), num_parallel_calls=4)
int_val_ds   = val_ds.map(  lambda x, y: (text_vectorization_seq(x), y), num_parallel_calls=4)

for inputs, targets in int_train_ds:
    print("Sequence batch shape:", inputs.shape, "← 32 samples, each padded to 600 integers")
    break

# shape=(None,) means variable-length sequences are accepted at inference time
inputs = keras.Input(shape=(None,), dtype="int64")

# Embedding → 256-dim dense vector
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256, mask_zero=True)(inputs)

# Bidirectional to run it forwards AND backwards,
x        = layers.Bidirectional(layers.LSTM(32))(embedded)
x        = layers.Dropout(0.5)(x)
outputs  = layers.Dense(1, activation="sigmoid")(x)

model_seq = keras.Model(inputs, outputs)
model_seq.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
model_seq.summary()

print("\nTraining Sequence model...")
model_seq.fit(int_train_ds, validation_data=int_val_ds, epochs=3)

Sequence batch shape: (32, 600) ← 32 samples, each padded to 600 integers


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  5,120,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 64)        │     73,984 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)


Training Sequence model...
Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.7684 - loss: 0.4746 - val_accuracy: 0.7304 - val_loss: 0.5205
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 33s 52ms/step - accuracy: 0.8745 - loss: 0.3112 - val_accuracy: 0.8852 - val_loss: 0.2959
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.9066 - loss: 0.2422 - val_accuracy: 0.8564 - val_loss: 0.3612


## Word Frequency in the IMDB Vocabulary

This cell loads the preprocessed IMDB dataset directly from Keras and counts how many times each word ID appears across all 25,000 training reviews — illustrating why high-frequency stop-words dominate raw counts and why TF-IDF normalization matters.

In [ ]:
from tensorflow.keras.datasets import imdb
from collections import Counter

print("Loading 25,000 IMDB training reviews...\n")

(x_train, _), _ = imdb.load_data()

# Flatten all reviews into list
all_ids = [word_id for review in x_train for word_id in review]
id_counts = Counter(all_ids)

# rank 1 = most frequent word in corpus
word_index = imdb.get_word_index()

# The stored IDs are offset by +3 to reserve space for special tokens
reverse_word_index = {rank + 3: word for word, rank in word_index.items()}
reverse_word_index.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

print("Top 10 most frequent words in the IMDB training set:")
for word_id, count in id_counts.most_common(10):
    word = reverse_word_index.get(word_id, "Unknown")
    print(f"  '{word}' (ID {word_id}) → repeated {count:,} times")

Loading 25,000 IMDB training reviews...

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Top 10 most frequent words in the IMDB training set:
  'the' (ID 4) → repeated 336,148 times
  'and' (ID 5) → repeated 164,097 times
  'a' (ID 6) → repeated 163,040 times
  'of' (ID 7) → repeated 145,847 times
  'to' (ID 8) → repeated 135,708 times
  'is' (ID 9) → repeated 107,313 times
  'br' (ID 10) → repeated 101,871 times
  'in' (ID 11) → repeated 93,934 times
  'it' (ID 12) → repeated 79,058 times
  'i' (ID 13) → repeated 77,142 times


## End-to-End Inference Model

During training, `TextVectorization` ran inside the `tf.data` pipeline (on CPU, asynchronously) for speed.

For **deployment**, we wrap the vectorization layer *inside* the model so it accepts raw strings directly — no separate preprocessing step needed in production. We then run the model on 50 real test reviews and print each prediction.

In [ ]:
string_inputs    = keras.Input(shape=(1,), dtype="string")
processed_inputs = text_vectorization_tfidf(string_inputs)
outputs          = model_bow(processed_inputs)         # standardize + tokenize + TF-IDF encode
inference_model  = keras.Model(string_inputs, outputs)

print("Running end-to-end inference on 50 real test reviews...\n")

# unbatch() removes the batch dimension so we can take individual reviews
fifty_reviews_ds = test_ds.unbatch().take(50)
sample_texts, sample_labels = [], []
for review, label in fifty_reviews_ds:
    sample_texts.append(review)
    sample_labels.append(label)

# Stack individual string tensors into a single tensor block
sample_reviews_tensor = tf.stack(sample_texts)
predictions = inference_model(sample_reviews_tensor)

for i in range(50):
    review_text          = sample_texts[i].numpy().decode('utf-8')
    true_sentiment       = "Positive" if sample_labels[i].numpy() == 1 else "Negative"
    predicted_confidence = float(predictions[i]) * 100
    predicted_sentiment  = "Positive" if predicted_confidence >= 50 else "Negative"

    print(f"Review {i+1}: '{review_text}'")
    print(f"True Label:     {true_sentiment}")
    print(f"Model Predicts: {predicted_sentiment} ({predicted_confidence:.2f}% positive confidence)")
    print("-" * 80)

Running end-to-end inference on 50 real test reviews...

Review 1: 'I will freely admit that I haven't seen the original movie, but I've read the play, so I've some background with the "original." If you shuck off the fact that this is a remake of an old classic, this movie is smart, witty, fresh, and hilarious. Yes, the casting decisions may seem strange, but they WORK. <br /><br />I'm a staunch feminist, and I wasn't offended in the slightest by this movie--despite what other women might be saying. This is NOT a movie for men to see (so please, ladies, don't drag your guys to see it with you, that's just cruel); women will get the jokes, the situations, and the relationships. <br /><br />I was pleasantly surprised by the depth that Annette Bening brought to her character...she did an excellent job. Debra Messing was adorable, and Candice Bergen was fantastic. I was less impressed by Meg Ryan...she brought emotion to the table, but her comedic take on it was less strong. The all-femal

# Transformers

## Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## 1. Transformer Encoder Layer

This class builds a Transformer Encoder block

In [ ]:
#This TransformerEncoder class learns relationships between words using self-attention, processes the information with a small neural network, and improves stability using residual connections and normalization.
class TransformerEncoder(layers.Layer):
  #we then setup layers
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)

        # Multi-head self-attention layer
        # num_heads = number of attention heads
        # key_dim = size of attention vectors
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # Small feed-forward neural network
        # Sequential means layers run one after another
        self.dense_proj = keras.Sequential([
            # First dense layer expands/features extraction
            layers.Dense(dense_dim, activation="relu"),
            # Second dense layer returns to original embedding size
            layers.Dense(embed_dim),
        ])

        # Normalize values after attention block
        self.layernorm_1 = layers.LayerNormalization()
        # Normalize values after dense block
        self.layernorm_2 = layers.LayerNormalization()

#  forward pass is the process where input data moves through the neural network to produce an output prediction.
    def call(self, inputs, mask=None):
      # Check if padding mask exists
        if mask is not None:

           # Change mask shape so attention layer can use it
            # Adds an extra dimension
            mask = mask[:, tf.newaxis, :]

        # Self-attention:
        # attention_mask tells model which tokens to ignore
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask
        )
        # Residual connection:
        # Add original input to attention output
        # Then normalize
        proj_input = self.layernorm_1(inputs + attention_output)

        # Pass result through feed-forward network
        proj_output = self.dense_proj(proj_input)

        # Second residual connection:
        # Add previous tensor to dense output
        # Then normalize again
        return self.layernorm_2(proj_input + proj_output)


## 2. Positional Embedding Layer

Create a custom layer for:
 1. Word embeddings
 2. Position embeddings

In [ ]:
#PositionalEmbedding class converts word IDs into vectors, adds position information so the Transformer understands word order, and creates a mask to ignore padding tokens.
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
       # Initialize parent Layer class
        super().__init__(**kwargs)


        # Embedding layer for words/tokens
        # input_dim = vocabulary size
        # output_dim = size of each word vector
        # Example:
        # word ID 25 -> [0.12, -0.44, 0.91, ...]
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )


        # Embedding layer for positions
        # position 0 -> vector
        # position 1 -> vector
        # position 2 -> vector
        # Helps Transformer understand word order
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim
        )

# Forward pass
#This call() method converts input word IDs and their positions into embeddings, then adds them together so the Transformer knows both word meaning and word order.
    def call(self, inputs):
        # Get sequence length from inpu
        # Example:
        # input shape = (batch_size, 150)
        # length = 150
        length = keras.ops.shape(inputs)[-1]
        # Create position indices
        # Example:
        # [0, 1, 2, 3, 4, ...]
        positions = keras.ops.arange(0, length, 1)
        # Convert word IDs into vectors
        # Example:
        # [12, 45, 78]
        # ->
        # [[...], [...], [...]]
        embedded_tokens = self.token_embeddings(inputs)
        # Convert position numbers into vectors
        # Example:
        # position 0 -> vector
        # position 1 -> vector
        embedded_positions = self.position_embeddings(positions)

        # Add word embeddings + position embeddings
        return embedded_tokens + embedded_positions


 # Create padding mask
    def compute_mask(self, inputs, mask=None):
         # Returns:
        # True  -> real token
        # False -> padding token (0)
        # Example:
        # [45, 12, 0, 0]
        # ->
        # [True, True, False, False]
        return keras.ops.not_equal(inputs, 0)

## 3. Load IMDB Data

In [ ]:
# Number of most frequent words to keep from the IMDB dataset.
max_tokens = 5000
# Every review will be cut or padded to this length.
# Shorter reviews get 0s added.
# Longer reviews are cut to 150 words.
sequence_length = 150
# Size of each word vector.
# Each word ID becomes a vector of length 64.
embed_dim = 64
# Number of attention heads in the Transformer.
# More heads can learn different word relationships.
num_heads = 2
# Size of the hidden Dense layer inside the Transformer block.
dense_dim = 32
# Number of samples processed at once during training.
batch_size = 64
# Number of times the model sees the training data.
epochs = 2


# x_train and x_test contain movie reviews as lists of word IDs.
# y_train and y_test contain labels:
# 0 = negative review
# 1 = positive review
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=max_tokens
)

# Keep only first 8000 training samples.
# This makes training faster for practice.
x_train = x_train[:8000]
y_train = y_train[:8000]

# Keep only first 2000 testing samples.
x_test = x_test[:2000]
y_test = y_test[:2000]

# Make all reviews same length
# Neural networks need equal-length inputs.
# pad_sequences makes every review length = sequence_length.
x_train = keras.utils.pad_sequences(x_train, maxlen=sequence_length)
x_test = keras.utils.pad_sequences(x_test, maxlen=sequence_length)

print(x_train.shape)
print(x_test.shape)

(8000, 150)
(2000, 150)


## 4. Build Transformer Model

In [ ]:
max_tokens = 5000        # Vocabulary size
sequence_length = 150    # Length of each input review
embed_dim = 64           # Size of word vectors
num_heads = 2            # Number of attention heads
dense_dim = 32           # Hidden size inside Transformer
batch_size = 64          # Training batch size
epochs = 2               # Number of training rounds

# Input layer
# Each input sample is a sequence of 150 integer word IDs.
inputs = keras.Input(shape=(sequence_length,), dtype="int64")

# Converts word IDs into vectors and adds position information.
x = PositionalEmbedding(
    sequence_length,
    max_tokens,
    embed_dim
)(inputs)

# Transformer Encoder
# Applies self-attention and feed-forward layers.
# This lets the model learn relationships between words.
x = TransformerEncoder(
    embed_dim,
    dense_dim,
    num_heads
)(x)
# Convert sequence to one vector
# Transformer output shape is:
# (batch_size, sequence_length, embed_dim)
# GlobalMaxPooling1D converts it to:
# (batch_size, embed_dim)
x = layers.GlobalMaxPooling1D()(x)

# Dropout
# Randomly turns off 50% of units during training.
# Helps reduce overfitting.
x = layers.Dropout(0.5)(x)

# Output layer
# One neuron because this is binary classification:
# 0 = negative review
# 1 = positive review
# Sigmoid gives probability between 0 and 1.
outputs = layers.Dense(1, activation="sigmoid")(x)

# Connect input layer to output layer.
model = keras.Model(inputs, outputs)

model.compile(
    optimizer="rmsprop",              # Optimizer updates model weights
    loss="binary_crossentropy",       # Loss for binary classification
    metrics=["accuracy"]              # Show accuracy during training
)

# Show model structure
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_encoder' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 150, 64)   │    329,600 │ input_layer_3[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 150)       │          0 │ input_layer_3[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 150, 64)   │     37,664 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ transformer_enco… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 367,329 (1.40 MB)

 Trainable params: 367,329 (1.40 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Train Model

In [ ]:
# Train the model
model.fit(
    x_train,
    # Correct labels:
    # 0 = negative
    # 1 = positive
    y_train,
    # More epochs:
    # + can improve learning
    # - may cause overfitting
    epochs=5,
    # Number of samples processed at once
    batch_size=batch_size,
    # It is only used to check model performance.
    validation_split=0.2
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 13s 26ms/step - accuracy: 0.5566 - loss: 0.8498 - val_accuracy: 0.7412 - val_loss: 0.5464
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7306 - loss: 0.5293 - val_accuracy: 0.8087 - val_loss: 0.4214
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8120 - loss: 0.4130 - val_accuracy: 0.8250 - val_loss: 0.3847
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8492 - loss: 0.3452 - val_accuracy: 0.8550 - val_loss: 0.3372
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8791 - loss: 0.2968 - val_accuracy: 0.8525 - val_loss: 0.3407


## 6. Evaluate

In [ ]:
# The model predicts review sentiment
# and compares predictions with real answers.
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8430 - loss: 0.3561
Test accuracy: 0.843


## 7. Let's test one review

In [ ]:
# IMDB dataset stores reviews as numbers.
# Example:
# 14 -> "this"
# 20 -> "movie"
word_index = keras.datasets.imdb.get_word_index()

# Function to convert text reviewinto model input
def encode_review(text):
     # Convert all letters to lowercase
    words = text.lower().split()
     # Store encoded word IDs
    encoded = []


    # Loop through each word
    for word in words:
      # Get integer ID for word
        index = word_index.get(word)

        # Check if:
        # 1. Word exists
        # 2. Word is inside vocabulary limit
        if index is not None and index < max_tokens:

          # Add +3 because IMDB reserves:
            # 0 = padding
            # 1 = start token
            # 2 = unknown word
            # 3 = unused
            encoded.append(index + 3)
        else:
          # Unknown word token
           # Used when word is not in vocabulary.
            encoded.append(2)


    # Pad sequence so it matches model input size
    # Example:
    # [45, 21, 87]
    # ->
    # [0, 0, 0, ..., 45, 21, 87]
    return keras.utils.pad_sequences(
        [encoded],
        maxlen=sequence_length
    )

# Example review
review = "this movie, I love it"

# Predict sentiment
# Convert review into numbers
encoded_review = encode_review(review)
# Output is probability between 0 and 1
prediction = model.predict(encoded_review)

# Display prediction
print(f"Positive probability: {float(prediction[0][0]):.3f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 894ms/step
Positive probability: 0.920
